# Project 3
Analysis of grades in high schools in Oslo for the time-period 2007/2008 to 2024/2025. We look at the exam grades, oral and written. The analysis is based on theese subjects:

**From Mathematics and Natural Sciences:**
- physics 1 and 2 
- mathematics R1 and R2 
- chemistry 1 and 2 

**From manditory subjects:**
- mathematics 1T, 1P and 2P
- natural science
- english
- social sciences
- geography
- history
- religion


In [11]:
import pandas as pd
import numpy as np

data = pd.read_csv("selektert_Karakterer_i_videregaaende_skole.csv", encoding='utf-16-le', sep="\t", skiprows=1)
columns_drop = ["FagNivaa", "EnhetNivaa", "Vurderingsfagnavn", "Nasjonaltkode", "Organisasjonsnummer", "Nasjonalt"]
data = data.drop(columns=columns_drop)

#print(data.columns)

/tmp/ipykernel_6939/1523042130.py:4: DtypeWarning: Columns (0: 2007-08.Muntlig eksamen.Alle eierformer.Alle kjønn.Snittkarakter, 1: 2007-08.Muntlig eksamen.Alle eierformer.Alle kjønn.Antall elever, 2: 2007-08.Muntlig eksamen.Alle eierformer.Gutt.Snittkarakter, 3: 2007-08.Muntlig eksamen.Alle eierformer.Gutt.Antall elever, 4: 2007-08.Muntlig eksamen.Alle eierformer.Jente.Snittkarakter, 5: 2007-08.Muntlig eksamen.Alle eierformer.Jente.Antall elever, 6: 2007-08.Skriftlig eksamen.Alle eierformer.Alle kjønn.Snittkarakter, 7: 2007-08.Skriftlig eksamen.Alle eierformer.Alle kjønn.Antall elever, 8: 2007-08.Skriftlig eksamen.Alle eierformer.Gutt.Snittkarakter, 9: 2007-08.Skriftlig eksamen.Alle eierformer.Gutt.Antall elever, 10: 2007-08.Skriftlig eksamen.Alle eierformer.Jente.Snittkarakter, 11: 2007-08.Skriftlig eksamen.Alle eierformer.Jente.Antall elever, 12: 2007-08.Standpunkt.Alle eierformer.Alle kjønn.Snittkarakter, 13: 2007-08.Standpunkt.Alle eierformer.Alle kjønn.Antall elever, 14: 2007-08.

#### Making a dataset consisting of only the schools in Oslo.

In [12]:
oslo_data = data[data['Fylkekode'] == 3]
#oslo_data.to_csv("selektert_oslo.csv", index=False, encoding="utf-8")

#### Making new names of columns
Shorter and easier to read, and easier to do data analysis on.

In [13]:
# Gir kolonnene nye og kortere lesbare navn

map_geographic = {
    "Vurderingsfagkode": "Fagkode",
    "Fylkekode": "Fylkekode",
    "Fylke": "Fylke",
    "EnhetNavn": "Skolenavn"
}

map_evaluation = {
    "Skriftlig eksamen": "SE",
    "Muntlig eksamen": "ME",
    "Standpunkt": "SP"
}

map_gender = {
    "Alle kjønn": "AK",
    "Gutt": "G",
    "Jente": "J"
}

map_grades = {
    "snittkarakter": "SK",
    "Antall elever": "AE",
    "Snittkarakter.Brudd": "SK.Brudd",
    "Antall elever.Brudd": "AE.Brudd"
}

def rename(col):
    if col in map_geographic:
        return map_geographic[col]
    
    parts = col.split(".")

    year = parts[0]
    type = parts[1]
    gender = parts[3]
    grade = ".".join(parts[4:]) # for the "Brudd"-stuff

    return f"{year}.{map_evaluation.get(type, type)}.{map_gender.get(gender, gender)}.{map_grades.get(grade, grade)}"

oslo_data.columns = [rename(c) for c in oslo_data.columns]

print(oslo_data.columns)


    


Index(['Fagkode', 'Fylkekode', 'Fylke', 'Skolenavn',
       '2007-08.ME.AK.Snittkarakter', '2007-08.ME.AK.AE',
       '2007-08.ME.G.Snittkarakter', '2007-08.ME.G.AE',
       '2007-08.ME.J.Snittkarakter', '2007-08.ME.J.AE',
       ...
       '2024-25.SE.G.Snittkarakter', '2024-25.SE.G.AE',
       '2024-25.SE.J.Snittkarakter', '2024-25.SE.J.AE',
       '2024-25.SP.AK.Snittkarakter', '2024-25.SP.AK.AE',
       '2024-25.SP.G.Snittkarakter', '2024-25.SP.G.AE',
       '2024-25.SP.J.Snittkarakter', '2024-25.SP.J.AE'],
      dtype='str', length=340)


#### Mapping the subjects to their respective code.
Just so it is easier for us to read.

In [15]:
fag = {
    "REA3004": "Fysikk 1", # 2008-2022
    "REA3038": "Fysikk 1", # 2022-
    "REA3005": "Fysikk 2", # 2009-2023
    "REA3039": "Fysikk 2", # 2023-
    "REA3022": "Matematikk R1", # 2008-2022
    "REA3056": "Matematikk R1", #2022-
    "REA3024": "Matematikk R2", # 2009-2023
    "REA3058": "Matematikk R2", #2023-
    "REA3011": "Kjemi 1", # 2008-2022
    "REA3045": "Kjemi 1", # 2022-
    "REA3012": "Kjemi 2", # 2009-2023
    "REA3046": "Kjemi 2", # 2023-
    "MAT1013": "Matematikk 1T", # 2010-H2021
    "MAT1021": "Matematikk 1T", # 2021- 
    "MAT1011": "Matematikk 1P", # 2010-H2021
    "MAT1019": "Matematikk 1P", # 2021-
    "MAT1015": "Matematikk 2P", # 2011-2022
    "MAT1023": "Matematikk 2P", # 2022-
    "ENG1002": "Engelsk, vg1 studieforberedende utdanningsprogram", # 2007-H2021
    "ENG1007": "Engelsk vg1 studieforberedende utdanningsprogram", # 2021
    "SAF1001": "Samfunnsfag", # 2007-H2022
    "SAK1001": "Samfunnskunnskap", # 2021-
    "GEO1001": "Geografi", # 2007-2021
    "GEO1003": "Geografi", # 2021-
    "HIS1002": "Historie vg3 studieforberedende utdanningsprogram", # 2009-2023
    "HIS1010": "Historie Vg3 studieforberedende utdanningsprogram", # 2023
    "REL1001": "Religion og etikk", # 2009-2023
    "REL1003": "Religion og etikk", # 2023-
    "NAT1002": "Naturfag, vg1 studieforberedende utdanningsprogram", # 2007-2021
    "NAT1007": "Naturfag vg1 studieforberedende utdanningsprogram", ## 2021
}

Be aware of 

*"Fra og med skoleåret 2012-2013, endret beregningsgrunnlag for karakterstatistikken, skriftlig eksamen."*